# Set up in cloud

In [1]:
!pip install -e .

Obtaining file:///home/jupyter/grouping-trainer
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for grouping-trainer (pyproject.toml) ... done
  Created wheel for grouping-trainer: filename=grouping_trainer-0.1.0-0.editable-py3-none-any.whl size=10048 sha256=705bf59db30fe56ea2dfc388e6ef3f095f0ee57fa751d7f2324e5abe94a76857
  Stored in directory: /var/tmp/pip-ephem-wheel-cache-6rnmmfvf/wheels/5f/ca/75/881118fa97e0f695c6b005134dc42a54df5074c95ebd09bef1
Successfully built grouping-trainer
  Attempting uninstall: grouping-trainer
    Found existing installation: grouping-trainer 0.1.0
    Uninstalling grouping-trainer-0.1.0:
      Successfully uninstalled grouping-trainer-0.1.0


In [10]:
!gsutil -m cp -r gs://seer-models/models/issue_grouping_v1 .

Copying gs://seer-models/models/issue_grouping_v1/.DS_Store...
Copying gs://seer-models/models/issue_grouping_v1/data.pkl...                   
Copying gs://seer-models/models/issue_grouping_v1/embeddings/1_Pooling/config.json...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/README.md...       
Copying gs://seer-models/models/issue_grouping_v1/embeddings/config.json...     
Copying gs://seer-models/models/issue_grouping_v1/embeddings/config_sentence_transformers.json...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/modules.json...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/configuration_bert.py...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/merges.txt...      
Copying gs://seer-models/models/issue_grouping_v1/embeddings/model.safetensors...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/sentence_bert_config.json...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/special_tokens_map.json...
Copyin

In [11]:
!gsutil -m -o GSUtil:check_hashes=never cp -r gs://grouping-data/final_csvs .

Copying gs://grouping-data/final_csvs/synthetic-semi-easy-negatives.csv...
Copying gs://grouping-data/final_csvs/test.csv...                               
Copying gs://grouping-data/final_csvs/train.csv...                              
Copying gs://grouping-data/final_csvs/val.csv...                                
/ [4/5 files][  5.4 GiB/  5.4 GiB]  99% Done  99.9 MiB/s ETA 00:00:00           

# Set up

In [1]:
token_value = !gcloud secrets versions access latest --secret=wandb-api-key --project=996102297610
import os

os.environ["WANDB_API_KEY"] = token_value[0]
del token_value

In [2]:
import wandb

For some reason, you need to run this next cell, interrupt it (it will hang), and then run it again (it will
immediately succeed)

In [4]:
wandb.login()

wandb: Currently logged in as: kush-dubey (sentry-seer) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
import math
import warnings

from datetime import datetime
from sentence_transformers import SentenceTransformerTrainingArguments
from sentence_transformers.training_args import MultiDatasetBatchSamplers
import torch

import grouping_trainer as gt
import utils

In [6]:
assert torch.cuda.is_available()

In [7]:
timestamp = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")

Some vars to care about

In [23]:
SAMPLE_TRAIN: int | None = 100_000 if torch.cuda.is_available() else 30
SAMPLE_VAL: int | None = 10_000 if torch.cuda.is_available() else 20

OUTPUT_DIR = f"./{timestamp}-output"
PER_DEVICE_TRAIN_BATCH_SIZE = 256
GRADIENT_ACCUMULATION_STEPS = 1
GRADIENT_CHECKPOINTING = True  # we need flash attn

PER_DEVICE_EVAL_BATCH_SIZE = 1
EVAL_STEPS = 80
PER_DEVICE_TOKEN_BUDGET = 8192 * 2  # tried increasing for A100 80GB

In [9]:
assert (EVAL_STEPS % 5) == 0, "pls for sanity make it divisible by 5"

# Load model

In [10]:
gt.utils._cuda_empty_cache()

In [11]:
model_path = "issue_grouping_v1/embeddings"
# model_path = "/Users/kdubey/projects/seer/models/issue_grouping_v1/embeddings"
model = gt.utils.SentenceTransformer(
    str(model_path),
    trust_remote_code=True,
    # model_kwargs=dict(
    #     dtype=torch.bfloat16,
    #     attn_implementation="sdpa",  # not possible for jina-ai :-(
    # )
)
model.device

/opt/conda/lib/python3.10/site-packages/torch/onnx/_internal/registration.py:162: OnnxExporterWarning: Symbolic function 'aten::scaled_dot_product_attention' already registered for opset 14. Replacing the existing function with new function. This is unexpected. Please report it on https://github.com/pytorch/pytorch/issues.
  warnings.warn(


device(type='cuda', index=0)

In [12]:
assert "layernorm" in repr(model[0].auto_model).lower()
assert "batch" not in repr(model[0].auto_model).lower()

Don't have batch norm. That could mess up stuff for the deduplication strategy.

In [13]:
_ = model.encode("test")

# Load data

We'll make a `dataset_val` for val loss.

In [14]:
dataset_val = gt.train.df_to_dataset(utils.load_val_df(sample_size=SAMPLE_VAL))
len(dataset_val)

10000

In [15]:
dataset_dict_train, frac_positive = utils.load_train_dataset_dict(
    sample_size=SAMPLE_TRAIN, min_dataset_size=PER_DEVICE_TRAIN_BATCH_SIZE
)
len(dataset_dict_train)

  0%|          | 0/144 [00:00<?, ?it/s]

109

In [17]:
sum(dataset_dict_train.num_rows.values())

100000

# Set up `Trainer`

In [21]:
evaluator = gt.evaluator.MinPrecisionEvaluator(
    sentences1=list(dataset_val["query_stacktrace_string"]),
    sentences2=list(dataset_val["candidate_stacktrace_string"]),
    labels=[int(record["label"]) for record in dataset_val],
    name="val",
    show_progress_bar=True,
    batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    truncate_dims=(64, 768),
)

Before training:

In [22]:
evaluator(model)

Batches:   0%|          | 0/6590 [00:00<?, ?it/s]

{'val_dim64_pr85_threshold': 0.9671444892883301,
 'val_dim64_pr85_precision': 0.8500604594921403,
 'val_dim64_pr85_recall': 0.7136020881670534,
 'val_dim64_pr85_n_predictions': 5789.0,
 'val_dim64_pr90_threshold': 0.9768604636192322,
 'val_dim64_pr90_precision': 0.9000916590284143,
 'val_dim64_pr90_recall': 0.5696055684454756,
 'val_dim64_pr90_n_predictions': 4364.0,
 'val_dim64_pr95_threshold': 0.9860374331474304,
 'val_dim64_pr95_precision': 0.9501125281320331,
 'val_dim64_pr95_recall': 0.36731438515081205,
 'val_dim64_pr95_n_predictions': 2666.0,
 'val_dim64_pr99_threshold': 0.9966863989830017,
 'val_dim64_pr99_precision': 0.9905838041431262,
 'val_dim64_pr99_recall': 0.07627610208816706,
 'val_dim64_pr99_n_predictions': 531.0,
 'val_dim768_pr85_threshold': 0.9609616994857788,
 'val_dim768_pr85_precision': 0.8500846023688663,
 'val_dim768_pr85_recall': 0.728538283062645,
 'val_dim768_pr85_n_predictions': 5910.0,
 'val_dim768_pr90_threshold': 0.9717447757720947,
 'val_dim768_pr90_pre

In [24]:
def init_bias(frac_positive: float):
    return math.log(frac_positive / (1 - frac_positive))

In [ ]:
trainer = gt.train.Trainer(
    model=model,
    args=SentenceTransformerTrainingArguments(
        # These should prolly be unchanged
        output_dir=OUTPUT_DIR,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=False,
        dataloader_pin_memory=torch.cuda.is_available(),
        num_train_epochs=1,
        # Save memory
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        #
        # Datalaoder
        multi_dataset_batch_sampler=MultiDatasetBatchSamplers.PROPORTIONAL,
        # Each iter, pick a project randomly, sample from it.
        # Next iter, pick another project randomly, sample from it, etc.
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        seed=42,  # passed to batch sampler
        #
        # Optimizer
        learning_rate=1e-4,
        learning_rate_mapping={
            # These are important to tune. Higher so that training doesn't get stuck. TODO: check
            r"^log_scale$": 2e-4,
            r"^bias$": 2e-4,
        },
        weight_decay=0.01,
        warmup_ratio=0.1,
        #
        # Eval
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        #
        # Logging
        logging_strategy="steps",
        logging_steps=EVAL_STEPS // 10,  # train loss alongside metrics table
        run_name=f"{timestamp}-grouping-trainer",
        report_to="wandb",
        #
        # Checkpointing
        save_strategy="steps",
        save_steps=EVAL_STEPS // 2,
        save_total_limit=2,
    ),
    #
    # Training
    loss=gt.train.SigmoidPairwiseLoss(
        model,
        bias_init=init_bias(frac_positive),
        log_of_scale_init=torch.tensor(5).log(),
        matryoshka_dims=[768, 512, 256, 128, 64],
        matryoshka_weights=[2, 1, 1, 0.5, 0.25],
        n_dims_per_step=2,
    ),
    data_collator=gt.train.DefaulDataCollator(tokenize_fn=model.tokenize),
    train_dataset=dataset_dict_train,
    shuffle_within_dataset=False,  # more cache hits in each forward
    per_device_token_budget=PER_DEVICE_TOKEN_BUDGET,
    #
    # Evaluator
    eval_dataset=dataset_val,  # val loss
    evaluator=evaluator,  # val recall at x precision
)

In [26]:
warnings.filterwarnings(
    "ignore",
    message=".*torch.utils.checkpoint: the use_reentrant parameter.*",
    category=UserWarning,
)

In [27]:
train_output = trainer.train()

You are using an old version of the checkpointing format that is deprecated (We will also silently ignore `gradient_checkpointing_kwargs` in case you passed it).Please update to the new format on your modeling file. To use the new format, you need to completely remove the definition of the method `_set_gradient_checkpointing` in your model.


Step,Training Loss,Validation Loss,Val Dim64 Pr85 Threshold,Val Dim64 Pr85 Precision,Val Dim64 Pr85 Recall,Val Dim64 Pr85 N Predictions,Val Dim64 Pr90 Threshold,Val Dim64 Pr90 Precision,Val Dim64 Pr90 Recall,Val Dim64 Pr90 N Predictions,Val Dim64 Pr95 Threshold,Val Dim64 Pr95 Precision,Val Dim64 Pr95 Recall,Val Dim64 Pr95 N Predictions,Val Dim64 Pr99 Threshold,Val Dim64 Pr99 Precision,Val Dim64 Pr99 Recall,Val Dim64 Pr99 N Predictions,Val Dim768 Pr85 Threshold,Val Dim768 Pr85 Precision,Val Dim768 Pr85 Recall,Val Dim768 Pr85 N Predictions,Val Dim768 Pr90 Threshold,Val Dim768 Pr90 Precision,Val Dim768 Pr90 Recall,Val Dim768 Pr90 N Predictions,Val Dim768 Pr95 Threshold,Val Dim768 Pr95 Precision,Val Dim768 Pr95 Recall,Val Dim768 Pr95 N Predictions,Val Dim768 Pr99 Threshold,Val Dim768 Pr99 Precision,Val Dim768 Pr99 Recall,Val Dim768 Pr99 N Predictions
80,0.422200,0.513786,0.629461,0.850108,0.856148,6945.000000,0.905047,0.900108,0.726508,5566.000000,0.989965,0.950192,0.431555,3132.000000,0.999307,0.990842,0.078451,546.000000,0.606428,0.850014,0.862094,6994.000000,0.899963,0.900089,0.732889,5615.000000,0.989032,0.950239,0.431990,3135.000000,0.999174,0.990625,0.091937,640.000000
160,0.299900,0.432710,0.385757,0.850084,0.875725,7104.000000,0.712223,0.900102,0.770882,5906.000000,0.937923,0.950023,0.598173,4342.000000,0.996405,0.990234,0.220563,1536.000000,0.378509,0.850112,0.876740,7112.000000,0.697634,0.900084,0.778567,5965.000000,0.932344,0.950126,0.599478,4351.000000,0.996257,0.990025,0.230278,1604.000000
240,0.267800,0.432595,0.307158,0.850014,0.886746,7194.000000,0.710432,0.900049,0.800464,6133.000000,0.958238,0.950090,0.612819,4448.000000,0.997751,0.990151,0.247825,1726.000000,0.260146,0.850076,0.889646,7217.000000,0.710062,0.900016,0.797564,6111.000000,0.954366,0.950167,0.619345,4495.000000,0.997535,0.990094,0.260876,1817.000000
320,0.213900,0.402861,0.299876,0.850028,0.890951,7228.000000,0.606070,0.900065,0.805829,6174.000000,0.893865,0.950011,0.611804,4441.000000,0.988537,0.990025,0.345418,2406.000000,0.246577,0.850075,0.899507,7297.000000,0.594157,0.900081,0.804669,6165.000000,0.880487,0.950066,0.623550,4526.000000,0.987133,0.990240,0.353103,2459.000000
400,0.218400,0.389177,0.352825,0.850076,0.897042,7277.000000,0.578800,0.900111,0.821926,6297.000000,0.853797,0.950000,0.650232,4720.000000,0.982294,0.990306,0.370360,2579.000000,0.317157,0.850041,0.902552,7322.000000,0.559930,0.900047,0.825261,6323.000000,0.846294,0.950116,0.654582,4751.000000,0.977247,0.990052,0.389646,2714.000000


Batches:   0%|          | 0/6590 [00:00<?, ?it/s]

Batches:   0%|          | 0/6590 [00:00<?, ?it/s]

Batches:   0%|          | 0/6590 [00:00<?, ?it/s]

Batches:   0%|          | 0/6590 [00:00<?, ?it/s]

Batches:   0%|          | 0/6590 [00:00<?, ?it/s]

In [28]:
trainer.save_model()

In [33]:
!gsutil -m cp -r wandb gs://grouping-data/runs/{OUTPUT_DIR}

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Skipping symlink directory "wandb/latest-run"
Copying file://wandb/debug.log [Content-Type=application/octet-stream]...
Copying file://wandb/debug-internal.log [Content-Type=application/octet-stream]...
Copying file://wandb/run-20251219_113514-5uvbi9fy/run-5uvbi9fy.wandb [Content-Type=application/octet-stream]...
Copying file://wandb/run-20251219_113514-5uvbi9fy/logs/debug.log [Content-Type=application/octet-stream]...
Copying file://wandb/run-20251219_113514-5uvbi9fy/logs/debug-core.log [Content-Type=application/octet-stream]...
Copying file://wandb/run-20251219_113514-5uvbi9fy/logs/debug-internal.log [Content-Type=application/octet-stream]...
Copying file://wandb/run-20251219_113514-5uvbi9fy/files/wandb-metadata.json [Content-Type=application/json]...
Copying file://wandb/run-20251219_113514-5uvbi9fy/files/requirements.txt [Content-Type=text/plain]...
Copying file://wandb/run-20251219_113514-5uvbi9fy/files/output.log [Content-Type=application/octet-stream]...
Copying file://wandb/run

In [29]:
!gsutil -m rsync -r {OUTPUT_DIR} gs://grouping-data/runs/{OUTPUT_DIR}/training

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Building synchronization state...
Starting synchronization...
Copying file://./2025-12-19-11-29-16-output/1_Pooling/config.json [Content-Type=application/json]...
Copying file://./2025-12-19-11-29-16-output/checkpoint-432/1_Pooling/config.json [Content-Type=application/json]...
Copying file://./2025-12-19-11-29-16-output/checkpoint-432/configuration_bert.py [Content-Type=text/x-python]...
Copying file://./2025-12-19-11-29-16-output/README.md [Content-Type=text/markdown]...
Copying file://./2025-12-19-11-29-16-output/checkpoint-432/config.json [Content-Type=application/json]...
Copying file://./2025-12-19-11-29-16-output/checkpoint-432/rng_state.pth [Content-Type=application/octet-stream]...
Copying file://./2025-12-19-11-29-16-output/checkpoint-432/README.md [Content-Type=text/markdown]...
Copying file://./2025-12-19-11-29-16-output/checkpoint-432/config_sentence_transformers.json [Content-Type=application/json]...
Copying file://./2025-12-19-11-29-16-output/checkpoint-432/merges.txt [

In [31]:
!gsutil -m cp -r train.ipynb gs://grouping-data/runs/{OUTPUT_DIR}

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Copying file://train.ipynb [Content-Type=application/octet-stream]...
/ [1/1 files][ 56.5 KiB/ 56.5 KiB] 100% Done                                    
Operation completed over 1 objects/56.5 KiB.                                     
